[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1ERiTP0XnWVnfjBhQtkXnrXroMuq7LgSf)

In [ ]:
!pip -q install stanza

In [ ]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
import stanza

In [ ]:
!wget -q -O data.csv https://raw.githubusercontent.com/IvoDz/lv-text-complexity/refs/heads/main/data/data.csv

In [ ]:
df = pd.read_csv("data.csv")
nlp = stanza.Pipeline('lv', processors='tokenize,pos,lemma', use_gpu=False)

In [ ]:
def stanza_tokenizer(text):
    doc = nlp(text)
    tokens = []
    for sentence in doc.sentences:
        for word in sentence.words:
            tokens.append(word.lemma)
    return tokens

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(df["level"])
X = df["text"]

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

# sagatavo tokenizētos datus uzreiz, iekš grid search ir lēnāk, sauc stanza_tokenizer vairākkārt
X_train = [' '.join(stanza_tokenizer(doc)) for doc in X_train_raw]
X_test = [' '.join(stanza_tokenizer(doc)) for doc in X_test_raw]

vectorizer = TfidfVectorizer()

In [ ]:
pipeline = Pipeline([
    ('tfidf', vectorizer),
    ('clf', LogisticRegression(max_iter=1000)),
])

param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2), (1,3)],
    'tfidf__min_df': [1, 3],
    'tfidf__max_df': [0.9, 1.0],
    'clf__C': [0.1, 1.0, 10]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    verbose=0,
    n_jobs=1,
)

grid.fit(X_train, y_train)

y_pred = grid.predict(X_test)
print("Best params:", grid.best_params_)
print("F1 (macro):", grid.best_score_)
print(classification_report(y_test, y_pred, target_names=le.classes_))